# Codebase Assistant — Recommended Demo

This notebook is the recommended end-to-end demonstration of the project.

It runs the complete multi-agent workflow against a small sample repository:

```
Repository
    ↓
Code Analysis Agent
    ↓
Documentation Agent
    ↓
Testing Agent
    ↓
Display outputs
```

The same agents are available from the primary CLI:

```bash
python app/main.py . --agent all
python app/main.py . --agent analysis --question "Find security bugs"
```

**Requirements:** a working `OPENROUTER_API_KEY` in `Project/.env`, and the project dependencies installed.

## 1. Imports

This notebook lives at the project root, next to `codebase_assistant/` and `app/`.

In [ ]:
from __future__ import annotations

import sys
import tempfile
import uuid
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "app") not in sys.path:
    sys.path.insert(0, str(ROOT / "app"))

from codebase_assistant.config import Config
from codebase_assistant.schemas.schemas import (
    AgentRequest,
    AgentType,
    DocumentationResult,
    TestingResult,
)
from codebase_assistant.supervisor import Supervisor
from report_formatter import (
    format_documentation_result,
    format_report,
    format_testing_result,
)

print("Imports ready.")
print("Project root:", ROOT)

## 2. Sample repository

A tiny local package keeps the demo fast while still exercising analysis, documentation, and test generation.

In [ ]:
repo = Path(tempfile.mkdtemp(prefix="codebase_assistant_demo_"))
sample_code = '"""Tiny math helpers used by the demo."""\n\n'
sample_code += "def add(a: int, b: int) -> int:\n"
sample_code += '    """Return the sum of two integers."""\n'
sample_code += "    return a + b\n\n\n"
sample_code += "def divide(a: float, b: float) -> float:\n"
sample_code += '    """Divide a by b. Raises ZeroDivisionError when b is 0."""\n'
sample_code += "    if b == 0:\n"
sample_code += '        raise ZeroDivisionError("b must be non-zero")\n'
sample_code += "    return a / b\n"
(repo / "math_utils.py").write_text(sample_code, encoding="utf-8")
(repo / "requirements.txt").write_text("pytest>=7.0\n", encoding="utf-8")

print("Sample repository:", repo)
print("Files:", sorted(path.name for path in repo.iterdir()))


## 3. Create the Supervisor

Creating the `Supervisor` wires OpenRouter, the RAG stack, the ToolRegistry, and all three agents.

In [ ]:
supervisor = Supervisor(config=Config.load())
analysis_agent = supervisor.agents[AgentType.CODE_ANALYSIS]
documentation_agent = supervisor.agents[AgentType.DOCUMENTATION]
testing_agent = supervisor.agents[AgentType.TESTING]

print("Supervisor ready.")
print("Registered tools:", len(supervisor.tool_registry.list_tools()))
print("Agents:", [agent_type.value for agent_type in supervisor.agents])

## 4. Code Analysis Agent

```
Repository → Code Analysis Agent → report
```

In [ ]:
analysis_report = analysis_agent.analyze_repository(
    repository_path=str(repo),
    question="Find likely bugs and correctness problems in this code.",
)

print(format_report(analysis_report, color=False))
print(
    f"Findings: {len(analysis_report.findings)} | "
    f"model_used={analysis_report.model_used} | "
    f"notes={len(analysis_report.notes)}"
)


## 5. Documentation Agent

```
Repository → Documentation Agent → README-style DocumentationResult
```

In [ ]:
documentation_response = documentation_agent.handle(
    AgentRequest(
        task_id=str(uuid.uuid4()),
        agent_type=AgentType.DOCUMENTATION,
        instruction="Generate a README summary for this repository.",
        context={
            "repo_path": str(repo),
            "repository_path": str(repo),
            "doc_type": "readme",
        },
    )
)

print("success:", documentation_response.success)
if documentation_response.errors:
    print("errors:", documentation_response.errors)

if isinstance(documentation_response.output, DocumentationResult):
    print(format_documentation_result(documentation_response.output))
else:
    print(documentation_response.output)

## 6. Testing Agent

```
Repository → Testing Agent → pytest modules
```

In [ ]:
testing_response = testing_agent.handle(
    AgentRequest(
        task_id=str(uuid.uuid4()),
        agent_type=AgentType.TESTING,
        instruction=(
            "Generate pytest unit tests for this repository, covering "
            "functions, methods, edge cases, invalid inputs, and "
            "common failure scenarios."
        ),
        context={
            "repo_path": str(repo),
            "repository_path": str(repo),
        },
    )
)

print("success:", testing_response.success)
if testing_response.errors:
    print("errors:", testing_response.errors)

if isinstance(testing_response.output, TestingResult):
    print(format_testing_result(testing_response.output, include_source=True))
else:
    print(testing_response.output)

## 7. Optional: one-shot goal through the Supervisor

`handle_goal` routes a natural-language request to the same agents in pipeline order and aggregates their responses.

In [ ]:
responses = supervisor.handle_goal(
    "analyze, document and generate tests",
    repo_path=str(repo),
)

for response in responses:
    print(
        f"{response.agent_type.value}: "
        f"success={response.success} "
        f"output={type(response.output).__name__} "
        f"errors={len(response.errors)}"
    )

## 8. CLI equivalents

From the project root:

```bash
# Interactive menu
python app/main.py

# Non-interactive
python app/main.py . --agent analysis
python app/main.py . --agent documentation
python app/main.py . --agent testing
python app/main.py . --agent all
python app/main.py . --agent analysis --question "Find security bugs"
```

`python -m codebase_assistant.main` still works, but it is deprecated and only forwards to `app/main.py`.